In [1]:
from primaite.agents.aegis.gllm import GLLM
from primaite.agents.aegis.modules.openai import OpenAIClient
from primaite.agents.git_agent import GITAgent
from torch.utils.data import DataLoader
from primaite.agents.llm.utils import network_connectivity_desc, obs_view_full
import logging

logging.disable(logging.CRITICAL)

/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
2024-08-29 10:59:19.429585: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-29 10:59:19.477914: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow wit

In [2]:
gllm = GLLM()
openai = OpenAIClient(openai_api_key="")

In [3]:
questions = [
    "What is the total number of nodes in the network?",
    "Describe the network",
    "What connects to CLIENT_1?",
    "What connects to the management console?",
    "What is connected to SWITCH_2?",
    "Which nodes are compromised if any?",
    "What is the hardware state of SWITCH_1?",
    "What is the software state of WEB_SERVER?",
    "Explain the network architecture, and what connects to what",
    "How many nodes connect to the management console?",
    "How many switches are there?",
    "How many unique node types are there?",
]

In [4]:
# Mock a primaite graph for development
agent = GITAgent(
    training_config_path="../src/primaite/config/_package_data/training/git.yaml",
    lay_down_config_path="../src/primaite/config/_package_data/lay_down/lay_down_config_6_data_manipulation.yaml",
)
obs = agent._env.reset()
data = agent.create_graph(obs)
network_desc = network_connectivity_desc(agent._env)
network_states = obs_view_full(agent.env_history[-1])

<Figure size 640x480 with 0 Axes>

In [5]:
import os
import pickle as pkl

if "openai_responses.pkl" not in os.listdir("./"):
    openai_responses = []
    openai_prompts = gllm.build_prompts(
        questions=questions, network_desc=network_desc + "\n" + network_states, model="openai"
    )
    for prompt in openai_prompts:
        openai_responses.append(openai.generate(prompt=prompt))

    with open("openai_responses.pkl", "wb") as file:
        pkl.dump(openai_responses, file)
else:
    with open("openai_responses.pkl", "rb") as file:
        openai_responses = pkl.load(file)

In [6]:
openai_responses

['The total number of nodes in the network is 9.',
 'Connected',
 'SWITCH_1',
 'SWITCH_1, SECURITY_SUITE, and BACKUP_SERVER',
 'WEB_SERVER, DATABASE_SERVER, BACKUP_SERVER',
 'None',
 'ON',
 'GOOD',
 'The network architecture consists of computer nodes (CLIENT_1 and CLIENT_2), switch nodes (SWITCH_1 and SWITCH_2), and server nodes (SECURITY_SUITE, MANAGEMENT_CONSOLE, WEB_SERVER, DATABASE_SERVER, and BACKUP_SERVER)',
 '1',
 '2',
 'There are 3 unique node types: COMPUTER, SWITCH, SERVER.']

In [7]:
from primaite.agents.aegis.data import GLLMDataset, collate_fn

dataset = GLLMDataset(
    graphs=[data for _ in range(len(questions))],
    questions=[question for question in questions],
    gt_answers=[response for response in openai_responses],
    llm=gllm.llm,
)

In [8]:
dataloader = DataLoader(dataset, batch_size=12, shuffle=True, collate_fn=collate_fn)

In [9]:
from primaite.agents.aegis.gllm import train_loop

gllm_responses = train_loop(
    model=gllm, dataloader=dataloader, network_desc=network_desc, n_epochs=2000, loss_fn="crossentropy"
)

Trainable LLM parameters: 25167873
All LLM parameters: 931237889
Percentage of trainable LLM parameters: 2.70%


/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


The network architecture consists of computer nodes (CLIENT_1 and CLIENT_2), switch nodes (SWITCH_1 and SWITCH_2), and server nodes (SECURITY_SUITE, MANAGEMENT_CONSOLE, WEB_SERVER, DATABASE_SERVER, and BACKUP_SERVER) <|im_end|>
The network architecture is the overall structure and organization of a computer network, including the devices, connections, and protocols that enable communication between them. It consists of several layers, each responsible for a specific function, such as:

1. Physical Layer (Layer 1): Defines the physical means of transmitting data, such as cables, Wi-Fi, or fiber optic.
2. Data Link Layer (Layer 2): Provides error-free transfer of data frames between devices on the same network


# Training notes
- Truncate GLLM response at GT max length to teach GLLM when it should have stopped generating.
- GLLM cosine embs loss function broke - fix so we can show that it wasn't viable in the report.